# TP 2-5 : Transformers et Large Language Models

**Objectif :** Comprendre l'architecture Transformer, utiliser BERT pour la classification de sentiment et explorer les mécanismes d'attention.

## Partie A — Prétraitement pour BERT

Cette partie consiste à préparer les textes pour leur traitement par le modèle BERT à l'aide du tokenizer de Hugging Face.

In [1]:
import torch
from transformers import BertTokenizer

print("PyTorch :", torch.__version__)
print("Transformers importé avec succès.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/ipyk

PyTorch : 2.2.2
Transformers importé avec succès.


In [2]:
from transformers import BertTokenizer

In [4]:
import requests
from pathlib import Path

model_dir = Path("bert-base-uncased")
model_dir.mkdir(exist_ok=True)

files = {
    "config.json": "https://huggingface.co/bert-base-uncased/resolve/main/config.json",
    "vocab.txt": "https://huggingface.co/bert-base-uncased/resolve/main/vocab.txt",
    "tokenizer_config.json": "https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json",
    "tokenizer.json": "https://huggingface.co/bert-base-uncased/resolve/main/tokenizer.json",
}

for filename, url in files.items():
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    
    with open(model_dir / filename, "wb") as f:
        f.write(response.content)
    
    print(filename, "OK")

config.json OK
vocab.txt OK
tokenizer_config.json OK
tokenizer.json OK


In [5]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

print("Tokenizer BERT chargé avec succès.")

Tokenizer BERT chargé avec succès.


In [6]:
text = "The movie was great but the ending was disappointing."

encoded = tokenizer(
    text,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encoded)

{'input_ids': tensor([[  101,  1996,  3185,  2001,  2307,  2021,  1996,  4566,  2001, 15640,
          1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [7]:
print("input_ids :")
print(encoded["input_ids"])

print("\nattention_mask :")
print(encoded["attention_mask"])

print("\ntoken_type_ids :")
print(encoded["token_type_ids"])

input_ids :
tensor([[  101,  1996,  3185,  2001,  2307,  2021,  1996,  4566,  2001, 15640,
          1012,   102]])

attention_mask :
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

token_type_ids :
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])


In [8]:
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])

print("Tokens :")
print(tokens)

Tokens :
['[CLS]', 'the', 'movie', 'was', 'great', 'but', 'the', 'ending', 'was', 'disappointing', '.', '[SEP]']


In [9]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

print("Tokenizer BERT chargé avec succès.")

Tokenizer BERT chargé avec succès.


In [10]:
import pandas as pd

df = pd.read_csv("data/IMDBDataset.csv")

print("Dimensions :", df.shape)
print(df.head())
print(df["sentiment"].value_counts())

Dimensions : (50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [11]:
df["label"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

print(df[["sentiment", "label"]].head())

  sentiment  label
0  positive      1
1  positive      1
2  positive      1
3  negative      0
4  positive      1


In [12]:
df_sample = df.sample(
    n=5000,
    random_state=42
).reset_index(drop=True)

print("Nombre total :", len(df_sample))
print(df_sample["sentiment"].value_counts())

Nombre total : 5000
sentiment
positive    2519
negative    2481
Name: count, dtype: int64


In [13]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_sample,
    test_size=2500,
    random_state=42,
    stratify=df_sample["label"]
)

print("Train :", len(train_df))
print("Test :", len(test_df))

Train : 2500
Test : 2500


In [14]:
train_encodings = tokenizer(
    train_df["review"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenisation du train terminée.")
print("Nombre d'exemples :", len(train_encodings["input_ids"]))

Tokenisation du train terminée.
Nombre d'exemples : 2500


In [15]:
test_encodings = tokenizer(
    test_df["review"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenisation du test terminée.")
print("Nombre d'exemples :", len(test_encodings["input_ids"]))

Tokenisation du test terminée.
Nombre d'exemples : 2500


In [16]:
print("Train input_ids :", len(train_encodings["input_ids"]))
print("Test input_ids :", len(test_encodings["input_ids"]))

print("Longueur d'une séquence :", len(train_encodings["input_ids"][0]))

Train input_ids : 2500
Test input_ids : 2500
Longueur d'une séquence : 128


In [17]:
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [18]:
train_dataset = IMDbDataset(
    train_encodings,
    train_df["label"].tolist()
)

test_dataset = IMDbDataset(
    test_encodings,
    test_df["label"].tolist()
)

print("Train dataset :", len(train_dataset))
print("Test dataset :", len(test_dataset))

Train dataset : 2500
Test dataset : 2500


### Chargement du modèle BERT

Nous utilisons le modèle `bert-base-uncased` pré-entraîné et ajoutons une
tête de classification binaire avec deux classes : négatif (0) et positif (1).

In [19]:
import requests

url = "https://huggingface.co/bert-base-uncased/resolve/main/config.json"

response = requests.get(url, timeout=20)

print("Status :", response.status_code)
print("Taille :", len(response.content))

Status : 200
Taille : 570


In [20]:
import requests

url = "https://huggingface.co/bert-base-uncased/resolve/main/pytorch_model.bin"

response = requests.get(
    url,
    stream=True,
    timeout=30
)

print("Status :", response.status_code)
print("Taille annoncée :", response.headers.get("content-length"))

Status : 200
Taille annoncée : 440473133


In [21]:
import requests
from pathlib import Path

model_dir = Path("bert-base-uncased")
model_dir.mkdir(exist_ok=True)

url = "https://huggingface.co/bert-base-uncased/resolve/main/pytorch_model.bin"
output_file = model_dir / "pytorch_model.bin"

print("Téléchargement de BERT...")

with requests.get(url, stream=True, timeout=60) as r:
    r.raise_for_status()
    
    total = int(r.headers.get("content-length", 0))
    downloaded = 0
    
    with open(output_file, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)
                
                if total:
                    percent = downloaded / total * 100
                    print(
                        f"\rProgression : {percent:.1f}%",
                        end=""
                    )

print("\nTéléchargement terminé.")

Téléchargement de BERT...
Progression : 100.0%
Téléchargement terminé.


In [22]:
from pathlib import Path

model_dir = Path("bert-base-uncased")

print("Dossier :", model_dir.exists())
print("Poids BERT :", (model_dir / "pytorch_model.bin").exists())
print("Taille :", (model_dir / "pytorch_model.bin").stat().st_size / 1024**2, "Mo")


Dossier : True
Poids BERT : True
Taille : 420.06791400909424 Mo


In [24]:
import requests
from pathlib import Path

model_dir = Path("bert-base-uncased")
model_dir.mkdir(exist_ok=True)

url = "https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors"
output_file = model_dir / "model.safetensors"

print("Téléchargement du modèle Safetensors...")

with requests.get(url, stream=True, timeout=60) as r:
    r.raise_for_status()

    total = int(r.headers.get("content-length", 0))
    downloaded = 0

    with open(output_file, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)

                if total:
                    print(
                        f"\rProgression : {downloaded / total * 100:.1f}%",
                        end=""
                    )

print("\nTéléchargement terminé.")

Téléchargement du modèle Safetensors...
Progression : 100.0%
Téléchargement terminé.


In [25]:
print((model_dir / "model.safetensors").exists())
print((model_dir / "model.safetensors").stat().st_size / 1024**2, "Mo")

True
420.0456314086914 Mo


In [26]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "./bert-base-uncased",
    num_labels=2
)

print("Modèle BERT chargé avec succès.")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ./bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Modèle BERT chargé avec succès.


In [27]:
from torch.utils.data import DataLoader

batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Nombre de batchs train :", len(train_loader))
print("Nombre de batchs test :", len(test_loader))

Nombre de batchs train : 313
Nombre de batchs test : 313


In [28]:
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Device utilisé :", device)

Device utilisé : cpu


In [29]:
model.to(device)

print("Modèle envoyé sur :", device)

Modèle envoyé sur : cpu


In [30]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

print("Optimiseur AdamW configuré.")

Optimiseur AdamW configuré.


In [31]:
epochs = 3

print("Nombre d'époques :", epochs)

Nombre d'époques : 3


In [32]:
model.train()

for batch in train_loader:
    batch = {
        key: value.to(device)
        for key, value in batch.items()
    }

    outputs = model(**batch)
    loss = outputs.loss

    print("Loss :", loss.item())
    break

Loss : 0.8781017661094666


In [33]:
from tqdm.auto import tqdm

train_losses = []
train_accuracies = []

for epoch in range(epochs):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Époque {epoch + 1}/{epochs}"
    )

    for batch in progress_bar:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

        accuracy = correct / total

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            accuracy=f"{accuracy:.4f}"
        )

    epoch_loss = total_loss / len(train_loader)
    epoch_accuracy = correct / total

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    print(
        f"Époque {epoch + 1}/{epochs} "
        f"- Loss : {epoch_loss:.4f} "
        f"- Accuracy : {epoch_accuracy:.4f}"
    )

Époque 1/3:   0%|          | 0/313 [00:00<?, ?it/s]

Époque 1/3 - Loss : 0.5260 - Accuracy : 0.7060


Époque 2/3:   0%|          | 0/313 [00:00<?, ?it/s]

Époque 2/3 - Loss : 0.2532 - Accuracy : 0.9060


Époque 3/3:   0%|          | 0/313 [00:00<?, ?it/s]

Époque 3/3 - Loss : 0.1127 - Accuracy : 0.9604


In [35]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = model(**batch)

        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

correct = sum(
    pred == label
    for pred, label in zip(all_predictions, all_labels)
)

accuracy = correct / len(all_labels)

print(f"Accuracy sur l'ensemble de test : {accuracy:.4f}")

Accuracy sur l'ensemble de test : 0.8564


In [36]:
from sklearn.metrics import classification_report

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["Négatif", "Positif"]
    )
)

              precision    recall  f1-score   support

     Négatif       0.85      0.87      0.86      1241
     Positif       0.87      0.85      0.86      1259

    accuracy                           0.86      2500
   macro avg       0.86      0.86      0.86      2500
weighted avg       0.86      0.86      0.86      2500



In [37]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_predictions)

print("Matrice de confusion :")
print(cm)

Matrice de confusion :
[[1075  166]
 [ 193 1066]]


# Partie C — Exploration de l'attention

Nous allons analyser les mécanismes d'attention de BERT à partir de la phrase :

> "The movie was great but the ending was disappointing."

Nous comparerons deux têtes d'attention de la dernière couche du modèle.

In [38]:
text_attention = "The movie was great but the ending was disappointing."

attention_inputs = tokenizer(
    text_attention,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

attention_inputs = {
    key: value.to(device)
    for key, value in attention_inputs.items()
}

print(attention_inputs)

{'input_ids': tensor([[  101,  1996,  3185,  2001,  2307,  2021,  1996,  4566,  2001, 15640,
          1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [39]:
model.eval()

with torch.no_grad():
    outputs_attention = model(
        **attention_inputs,
        output_attentions=True
    )

attentions = outputs_attention.attentions

print("Nombre de couches :", len(attentions))
print("Forme de la dernière couche :", attentions[-1].shape)

BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Nombre de couches : 12
Forme de la dernière couche : torch.Size([1, 12, 12, 12])


In [40]:
tokens_attention = tokenizer.convert_ids_to_tokens(
    attention_inputs["input_ids"][0]
)

print(tokens_attention)

['[CLS]', 'the', 'movie', 'was', 'great', 'but', 'the', 'ending', 'was', 'disappointing', '.', '[SEP]']


In [41]:
attention_head_0 = attentions[-1][0, 0].cpu()

print("Dimensions :", attention_head_0.shape)

Dimensions : torch.Size([12, 12])


In [44]:
ending_index = tokens_attention.index("ending")

attention_head_0 = attentions[-1][0, 0]

ending_attention = attention_head_0[ending_index]

for token, score in zip(
    tokens_attention,
    ending_attention.tolist()
):
    print(f"{token:15s} {score:.4f}")

[CLS]           0.0428
the             0.0201
movie           0.0428
was             0.0209
great           0.0238
but             0.0152
the             0.0223
ending          0.0279
was             0.0219
disappointing   0.0371
.               0.5930
[SEP]           0.1321


In [45]:
scores = list(zip(tokens_attention, ending_attention.tolist()))

scores_sorted = sorted(
    scores,
    key=lambda x: x[1],
    reverse=True
)

print("Attention de 'ending' :")

for token, score in scores_sorted[:5]:
    print(f"{token:15s} {score:.4f}")

Attention de 'ending' :
.               0.5930
[SEP]           0.1321
[CLS]           0.0428
movie           0.0428
disappointing   0.0371


In [46]:
attention_head_5 = attentions[-1][0, 5]

ending_attention_5 = attention_head_5[ending_index]

scores_5 = list(
    zip(tokens_attention, ending_attention_5.tolist())
)

scores_5_sorted = sorted(
    scores_5,
    key=lambda x: x[1],
    reverse=True
)

print("Attention de 'ending' — tête 5 :")

for token, score in scores_5_sorted[:5]:
    print(f"{token:15s} {score:.4f}")

Attention de 'ending' — tête 5 :
.               0.2447
[SEP]           0.1525
great           0.1519
ending          0.1295
disappointing   0.0640


### Comparaison entre la tête 0 et la tête 5

Les deux têtes d'attention présentent des comportements différents pour le
mot « ending ».

Dans la tête 0, l'attention est très fortement concentrée sur le token « . »
(score de 0,5930), suivi de « [SEP] » (0,1321). Le mot « disappointing »
obtient seulement un score de 0,0371.

Dans la tête 5, l'attention est plus répartie. Le token « . » reste le plus
important avec un score de 0,2447, mais « great » reçoit également une
attention importante (0,1519), ainsi que « ending » lui-même (0,1295) et
« disappointing » (0,0640).

On observe donc que les différentes têtes peuvent capturer des relations
différentes entre les tokens. La tête 0 semble davantage concentrée sur la
ponctuation finale, tandis que la tête 5 prend davantage en compte certains
mots du contexte, notamment « great », « ending » et « disappointing ».

## D14 — Comparaison des architectures Transformer

| Architecture | Principe d'attention | Entraînement / objectif | Applications principales |
|---|---|---|---|
| **Encoder-only (BERT)** | Attention bidirectionnelle : chaque token peut prendre en compte le contexte à gauche et à droite | Masked Language Modeling (MLM) | Classification, analyse de sentiment, NER |
| **Decoder-only (GPT)** | Attention masquée : un token ne peut regarder que les tokens précédents | Génération auto-régressive | Génération de texte, conversation, complétion |
| **Encoder-Decoder (T5)** | L'encodeur traite l'entrée et le décodeur génère la sortie avec une attention croisée | Apprentissage texte-à-texte | Traduction, résumé, génération conditionnelle |

## D15 — Pourquoi le positional encoding est-il indispensable dans le Transformer ?

Le Transformer traite les tokens d'une séquence en parallèle grâce au mécanisme
d'attention. Contrairement aux architectures récurrentes, l'attention seule ne
contient pas directement l'information sur l'ordre des mots.

Le positional encoding permet donc d'ajouter à chaque représentation de token
une information indiquant sa position dans la séquence.

Par exemple, les phrases :

- « Le chat mange la souris »
- « La souris mange le chat »

contiennent les mêmes mots, mais leur ordre change complètement le sens.

Les LSTM, quant à eux, traitent les éléments séquentiellement : le modèle reçoit
un token après l'autre et son état caché conserve une information provenant des
étapes précédentes. L'ordre est donc naturellement intégré au fonctionnement
récurrent du LSTM.

Ainsi, le positional encoding est indispensable au Transformer pour que le
modèle puisse distinguer les différentes positions des tokens et prendre en
compte l'ordre de la séquence.

## D16 — Rôle du facteur √dk dans le Scaled Dot-Product Attention

Dans le mécanisme d'attention, les scores sont calculés à partir du produit
scalaire entre les vecteurs Query (Q) et Key (K).

Le produit scalaire est ensuite divisé par la racine carrée de la dimension des
vecteurs Key :

Attention(Q,K,V) = softmax(QKᵀ / √dk)V

Le facteur √dk permet de maintenir les valeurs des scores d'attention dans une
plage raisonnable lorsque la dimension dk augmente.

Sans cette mise à l'échelle, les produits scalaires peuvent devenir très grands.
La fonction softmax produirait alors des probabilités très concentrées autour
d'une seule valeur, ce qui entraînerait des gradients très faibles et rendrait
l'apprentissage plus difficile.

Le facteur √dk stabilise donc les scores avant le passage dans la fonction
softmax et permet un entraînement plus stable du Transformer.